# Assignment 3: Continuous-Time Dynamical Systems Arbitrage
**Team:** Corentin Lepla & Shubham Nanewar

### 1. The Economics of the Strategy (Max 150 Words)
Standard pairs trading relies on static industry classifications and linear ordinary least squares (OLS). Our strategy arbitrages transient liquidity imbalances across a dynamically mapped supply-chain manifold. We reject static labels, utilizing the Hoberg-Phillips Textual Network Industry Classifications (TNIC) 10-K database to project target assets onto a continuous product-market subspace. 

Recognizing that supply-chain fundamentals drift while market microstructure is contaminated by bid-ask bounce, we extract the structural cointegrating tether using an Adaptive Kalman Filter. We hypothesize that smaller constituents within large indices suffer from finite liquidity; thus, temporary institutional flows create discontinuous price gaps (overreactions). By modeling this spread as a Jump-Diffusion SDE and applying a Hamilton-Jacobi-Bellman (HJB) optimal stopping boundary, we isolate and trade strictly when the stochastic pull of the fundamental mean physically overcomes the deterministic transaction friction.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import refinitiv.data as rd
from scipy import sparse
from scipy.optimize import minimize # for MLE
from numba import njit
from typing import Dict          
import pandas_datareader.data as web    
import warnings
warnings.filterwarnings('ignore')

### 2. Strategy Setup & Free Parameter Justification

Our engine operates on a daily "Tumbling Window" sequence, strictly separating fundamental calibration from physical execution.

* **Topological Breadth ($top\_n = 4$):** Projecting the target asset onto its 4 closest TNIC textual peers provides sufficient rank to span the synthetic subspace without introducing deep-tail illiquidity noise.
* **The Filtration Window ($window = 63$):** Standard rolling windows contaminate discrete corporate data releases. We use a 63-day (one trading quarter) window to strictly bind our AR(1) mean-reversion parameter ($\theta$) and Jump-Diffusion volatility ($\sigma_{MAD}$) to the current 10-Q earnings regime.
* **The ADF "Kill Switch" ($p < 0.05$):** We dynamically test the spread for stationarity. If the regime breaks into an $I(1)$ random walk, the algorithm mathematically aborts all trades to prevent infinite friction bleed.
* **Physical Friction:** We explicitly model $0.0005$ (5 bps) slippage to cross the spread and $0.02 / 252$ daily prime-broker borrow fees.
* **HJB Execution Boundary:** We abandon static z-scores in favor of a finite-difference PDE grid solver that calculates the exact boundary where expected mean-reversion profit exceeds friction costs.


## Strategy Setup, Asset Selection, & Parameter Justification

**Asset Selection (Why XOM.N?):** We anchor the strategy on Exxon Mobil (XOM.N). We explicitly target the energy sector because physical commodities possess rigid thermodynamic and supply-chain constraints. Unlike purely digital software firms, refineries and oil pipelines cannot pivot operations overnight. This physical rigidity enforces a slower, more robust structural cointegration ($\beta_t$) with competitors, making it an ideal candidate for state-space modeling.

**Topological Breadth ($top\_n = 4$):** Standard pairs trading pairs exactly two assets, leading to severe idiosyncratic risk. We project the target onto a synthetic index of its 4 closest TNIC textual peers. A $K=4$ dimensionality provides sufficient rank to span the localized supply-chain subspace while excluding deep-tail, illiquid micro-cap noise that would violate our execution friction assumptions.

**Kalman Filter Calibration (MLE):** We refuse to use heuristic guesses for the state-transition covariance ($Q$) and observation variance ($R$). We implement Maximum Likelihood Estimation (MLE) on the burn-in period to mathematically minimize the negative log-likelihood of the forecast innovation. The data itself dictates the physical signal-to-noise ratio.

**The Filtration Window ($window = 63$):** Standard rolling windows contaminate discrete corporate data releases. We use a 63-day (one trading quarter) Tumbling Window to strictly bind our AR(1) mean-reversion speed ($\theta$) and Jump-Diffusion volatility ($\sigma_{MAD}$) to the current 10-Q earnings regime.

**HJB Optimal Stopping:** Standard algorithms use arbitrary static z-scores (e.g., $\pm 2.0$) for entry boundaries. This ignores the physics of execution. We utilize a Finite Difference Method to solve the Hamilton-Jacobi-Bellman (HJB) Variational Inequality. The PDE dynamically calculates the exact execution boundary by equating the expected stochastic profit of the OU drift against the deterministic opportunity cost of time and physical transaction friction (5 bps slippage, 2% borrow).

In [5]:
class TNICUniverse:
    def __init__(self, tnic_data_path: str, target_year: int):
        """
        Initializes the TNIC network for a specific year to conserve RAM.
        """
        self.tnic_data_path = tnic_data_path
        self.target_year = target_year
        self.network = self._load_network_memory_safe()
        
        # We will populate this mapping dictionary later (RIC -> GVKEY)
        self.ric_to_gvkey: Dict[str, str] = {}

    def get_top_n_peers(self, target_ric: str, ric_to_gvkey: dict, gvkey_to_ric: dict, n_peers: int = 10):
        """
        Dynamically searches the TNIC textual graph using GVKEYs, and translates 
        the structurally closest peers back into executable RICs.
        """
        if self.network is None:
            self.load_data()
            
        target_gvkey = str(ric_to_gvkey.get(target_ric))
        if not target_gvkey or target_gvkey == 'None':
            raise ValueError(f"Target RIC {target_ric} not found in crosswalk dictionary.")
            
        # Search the graph using the correct GVKEY
        peers_1 = self.network[self.network['gvkey1'] == target_gvkey][['gvkey2', 'score']].rename(columns={'gvkey2': 'peer'})
        peers_2 = self.network[self.network['gvkey2'] == target_gvkey][['gvkey1', 'score']].rename(columns={'gvkey1': 'peer'})
        
        all_peers = pd.concat([peers_1, peers_2]).sort_values(by='score', ascending=False)
        
        # Filter the graph to ONLY keep peers that we can translate back into RICs for execution
        all_peers['peer_ric'] = all_peers['peer'].astype(str).map(gvkey_to_ric)
        mapped_peers = all_peers.dropna(subset=['peer_ric'])
        
        # Keep the Top N highest similarity scores
        top_n = mapped_peers.head(n_peers)
        peer_rics = top_n['peer_ric'].tolist()
        raw_scores = top_n['score'].values
        
        if len(peer_rics) == 0:
            raise ValueError(f"No mapped peers found for {target_ric}.")
        
        # Normalize the scores so they sum to 1.0 (L1 Norm) to create an index weight vector
        normalized_weights = raw_scores / np.sum(raw_scores)
        
        print(f"Dynamically mapped {len(peer_rics)} executable TNIC peers for {target_ric}.")
        return peer_rics, normalized_weights
        
    def _load_network_memory_safe(self) -> pd.DataFrame:
        """
        Streams the massive text file and only keeps rows for the target year.
        Uses optimized dtypes to prevent memory overflow.
        """
        print(f"Streaming TNIC database for year {self.target_year}...")
        
        # Define strict types to save RAM (strings for IDs, float32 for weights)
        dtypes = {
            'year': np.int16,
            'gvkey1': str,
            'gvkey2': str,
            'score': np.float32
        }
        
        # Read in chunks (since it's too big for Excel, it's big for memory)
        # Assuming the file is comma or tab separated. If tab, use sep='\t'
        chunk_iter = pd.read_csv(
            self.tnic_data_path, 
            sep=None,          # Auto-detect separator (tab or comma)
            engine='python',   
            dtype=dtypes,
            chunksize=100_000
        )
        
        filtered_chunks = []
        for chunk in chunk_iter:
            # Filter the chunk down to just our year before saving it
            target_chunk = chunk[chunk['year'] == self.target_year]
            filtered_chunks.append(target_chunk)
            
        network_df = pd.concat(filtered_chunks, ignore_index=True)
        print(f"Loaded {len(network_df)} peer connections for {self.target_year}.")
        return network_df

    def get_peer_weights(self, target_gvkey: str) -> pd.Series:
        """
        Extracts the similarity vector (w_i) for a specific firm.
        Returns a Pandas Series mapping gvkey2 -> score.
        """
        peers = self.network[self.network['gvkey1'] == target_gvkey]
        if peers.empty:
            raise ValueError(f"Firm {target_gvkey} not found in TNIC network.")
            
        # Return a series where the index is the peer GVKEY and the value is the score
        return peers.set_index('gvkey2')['score']

    def construct_peer_index(self, similarity_scores: np.ndarray, peer_prices: np.ndarray) -> np.ndarray:
        """
        Projects the peer prices onto the normalized weight vector.
        similarity_scores: (N,)
        peer_prices: (T, N)
        """
        assert similarity_scores.shape[0] == peer_prices.shape[1], "Dimension mismatch"
        assert np.all(similarity_scores >= 0), "Scores must be non-negative"
        
        total_weight = np.sum(similarity_scores)
        if total_weight == 0:
            raise ValueError("All similarity scores are zero.")
            
        normalized_weights = similarity_scores / total_weight
        
        # Matrix multiplication: (T, N) @ (N,) -> (T,)
        synthetic_index = np.einsum('ti,i->t', peer_prices, normalized_weights)
        return synthetic_index

# ----------------------------------------------------------------------------------------------------------------------------------------- #
class MidTermGovernor:
    def __init__(self, dt: float = 1/252):
        self.dt = dt
        self.alpha_t = None
        self.beta_t = None
        self.e_t = None       
        self.S_t = None       
        
        # We will dynamically overwrite these via MLE
        self.opt_q = 1e-4 
        self.opt_r = 1e-3 

    def _kalman_negative_log_likelihood(self, params, target_prices, peer_index):
        """
        The Objective Function.
        We optimize the natural log of the parameters to strictly enforce positive variances.
        """
        q = np.exp(params[0])
        r = np.exp(params[1])
        
        T = len(target_prices)
        Q = np.eye(2) * q
        P = np.zeros((2, 2))
        theta = np.zeros(2) 
        
        nll = 0.0
        
        for t in range(T):
            x_t = np.array([1.0, peer_index[t]])
            y_t = target_prices[t]
            
            # A priori prediction
            P = P + Q
            y_hat = np.dot(x_t, theta)
            e_t = y_t - y_hat
            
            # System Variance
            S_t = np.dot(x_t, np.dot(P, x_t)) + r
            
            # Accumulate the log-likelihood (skip first 10 days for numerical stabilization)
            if t > 10:
                nll += 0.5 * (np.log(S_t) + (e_t**2) / S_t)
                
            # A posteriori update
            K_t = np.dot(P, x_t) / S_t
            theta = theta + K_t * e_t
            
            # Joseph stabilized covariance update
            I_minus_Kx = np.eye(2) - np.outer(K_t, x_t)
            P = I_minus_Kx @ P @ I_minus_Kx.T + np.outer(K_t, K_t) * r
            
        return nll

    def calibrate_mle(self, target_prices: np.ndarray, peer_index: np.ndarray):
        """
        Finds the physical parameters of the structural relationship using 
        Maximum Likelihood on the burn-in sample (Year 1).
        """
        # Restrict calibration to the first 252 days to prevent look-ahead bias
        calib_len = min(252, len(target_prices))
        t_calib = target_prices[:calib_len]
        p_calib = peer_index[:calib_len]
        
        # Initial guess in log-space
        init_guess = [np.log(1e-4), np.log(1e-3)]
        
        # L-BFGS-B minimizes the negative log-likelihood
        res = minimize(self._kalman_negative_log_likelihood, init_guess, 
                       args=(t_calib, p_calib), method='L-BFGS-B')
        
        self.opt_q = np.exp(res.x[0])
        self.opt_r = np.exp(res.x[1])
        print(f"[MLE Calibration] Structural State Noise (Q): {self.opt_q:.2e} | Observation Noise (R): {self.opt_r:.2e}")

    def fit_kalman_cointegration(self, target_prices: np.ndarray, peer_index: np.ndarray) -> np.ndarray:
        # 1. Calibrate the physics engine automatically
        self.calibrate_mle(target_prices, peer_index)
        
        T = len(target_prices)
        Q_base = np.eye(2) * self.opt_q
        R_base = self.opt_r
        
        P = np.zeros((2, 2))
        theta = np.zeros(2) 
        
        self.alpha_t = np.zeros(T)
        self.beta_t = np.zeros(T)
        self.e_t = np.zeros(T)
        self.S_t = np.zeros(T)
        spread = np.zeros(T)
        
        target_returns = np.diff(target_prices, prepend=target_prices[0])
        
        for t in range(T):
            x_t = np.array([1.0, peer_index[t]])
            y_t = target_prices[t]
            
            # --- Roll's Model: Dynamic Observation Noise (R_t) ---
            if t > 5:
                cov_matrix = np.cov(target_returns[t-5:t], target_returns[t-6:t-1])
                roll_variance = max(0, -cov_matrix[0, 1])
            else:
                roll_variance = 0.0
            V_v = R_base + roll_variance
            
            # --- ADAPTIVE REGIME SHIFT LOGIC ---
            Q_t = Q_base.copy()
            if t > 20: 
                normalized_sq_error = (self.e_t[t-1]**2) / self.S_t[t-1]
                if normalized_sq_error > 9.0: 
                    # If structurally broken, inject a massive multiple of the baseline variance
                    Q_t = Q_t + np.eye(2) * (self.opt_q * 100)
            
            # Standard Prediction & Update
            P = P + Q_t
            y_hat = np.dot(x_t, theta)
            e_t = y_t - y_hat
            
            S_t = np.dot(x_t, np.dot(P, x_t)) + V_v
            K_t = np.dot(P, x_t) / S_t
            
            theta = theta + K_t * e_t
            I_minus_Kx = np.eye(2) - np.outer(K_t, x_t)
            P = I_minus_Kx @ P @ I_minus_Kx.T + np.outer(K_t, K_t) * V_v
            
            self.alpha_t[t] = theta[0]
            self.beta_t[t] = theta[1]
            self.e_t[t] = e_t
            self.S_t[t] = S_t
            spread[t] = y_t - (theta[0] + theta[1] * peer_index[t])
            
        return spread

    def fit_rolling_ou_process(self, local_spread: np.ndarray):
        Z_t = local_spread[1:]
        Z_t_minus_1 = local_spread[:-1]
        
        X = np.vstack([np.ones(len(Z_t_minus_1)), Z_t_minus_1]).T
        
        try:
            params = np.linalg.inv(X.T @ X) @ X.T @ Z_t
            a, b = params[0], params[1]
        except np.linalg.LinAlgError:
            return 0.0, 0.0, np.inf 
        
        if b >= 1 or b <= 0:
            return 0.0, 0.0, np.inf 
            
        theta_param = -np.log(b) / self.dt
        mu = a / (1 - b)
        
        # Revert to unscaled Robust MAD calibration because the Kalman output is now highly accurate
        epsilon = Z_t - (a + b * Z_t_minus_1)
        mad = np.median(np.abs(epsilon - np.median(epsilon)))
        robust_std = 1.4826 * mad + 1e-8 
        sigma = robust_std / np.sqrt(self.dt)
        
        half_life = np.log(2) / theta_param
        
        return mu, sigma, half_life

# --------------------------------------------------------------------------------- # 

def psor_solver(A_data, A_indices, A_indptr, b, obstacle, V_guess, omega=1.2, tol=1e-6, max_iter=1000):
    """
    Projected Successive Over-Relaxation (PSOR) to solve the Linear Complementarity Problem (LCP).
    This physically finds the "Free Boundary" (the optimal execution threshold).
    """
    n = len(b)
    V = V_guess.copy()
    
    for iteration in range(max_iter):
        error = 0.0
        for i in range(n):
            # Extract row i from CSR matrix
            row_start = A_indptr[i]
            row_end = A_indptr[i+1]
            
            sigma_sum = 0.0
            diag = 1.0
            for j_idx in range(row_start, row_end):
                j = A_indices[j_idx]
                val = A_data[j_idx]
                if i == j:
                    diag = val
                else:
                    sigma_sum += val * V[j]
                    
            # Gauss-Seidel step
            v_new = (b[i] - sigma_sum) / diag
            
            # Successive Over-Relaxation
            v_sor = V[i] + omega * (v_new - V[i])
            
            # The Projection (The "American Option" Early Exercise constraint)
            v_proj = max(v_sor, obstacle[i])
            
            error += abs(v_proj - V[i])
            V[i] = v_proj
            
        if error < tol:
            break
            
    return V
class HJBOptimalStopping:
    def __init__(self, z_min=-5.0, z_max=5.0, grid_points=1000):
        self.z_min = z_min
        self.z_max = z_max
        self.N = grid_points
        self.dz = (z_max - z_min) / (grid_points - 1)
        self.Z_grid = np.linspace(z_min, z_max, grid_points)

    def solve_entry_boundary(self, mu, theta, sigma, friction, discount_rate=0.05):
        """
        Builds the massive Finite Difference tridiagonal matrix for the OU process
        and solves the HJB Variational Inequality to find the optimal entry spread.
        """
        # 1. Construct the Infinitesimal Generator Matrix (L) for the OU process
        # L(V) = 0.5 * sigma^2 * V'' + theta * (mu - Z) * V'
        
        diag = np.zeros(self.N)
        lower = np.zeros(self.N - 1)
        upper = np.zeros(self.N - 1)
        
        sig2 = sigma**2
        dz2 = self.dz**2
        
        for i in range(1, self.N - 1):
            z = self.Z_grid[i]
            drift = theta * (mu - z)
            
            # Central difference for V' and V''
            lower[i-1] = (sig2 / (2 * dz2)) - (drift / (2 * self.dz))
            diag[i]   = -(sig2 / dz2) - discount_rate
            upper[i]   = (sig2 / (2 * dz2)) + (drift / (2 * self.dz))
            
        # Dirichlet boundary conditions (Value is zero at extreme infinities if we don't trade)
        diag[0] = 1.0
        diag[-1] = 1.0
        upper[0] = 0.0
        lower[-1] = 0.0

        A = sparse.diags([lower, diag, upper], offsets=[-1, 0, 1], format='csr')
        
        # 2. Define the Obstacle (The physical payoff of crossing the spread)
        # If we go LONG: We expect the spread to revert from Z_grid to mu, minus friction
        long_payoff = (mu - self.Z_grid) - friction
        
        # 3. Solve the Linear Complementarity Problem via PSOR
        b = np.zeros(self.N)
        V_guess = np.maximum(long_payoff, 0)
        
        V_optimal = psor_solver(A.data, A.indices, A.indptr, b, long_payoff, V_guess)
        
        # 4. Locate the Free Boundary
        # The optimal execution threshold is the exact coordinate where the Value Function 
        # is physically tangent to the Payoff function.
        exercise_region = np.isclose(V_optimal, long_payoff, atol=1e-4)
        
        # The boundary is the point closest to the mean where exercise is optimal
        try:
            boundary_idx = np.where(exercise_region & (self.Z_grid < mu))[0][-1]
            optimal_z_long = self.Z_grid[boundary_idx]
        except IndexError:
            optimal_z_long = -np.inf # Friction is too high; mathematically impossible to profit
            
        return optimal_z_long

In [7]:
def run_master_backtest(target_ric="XOM.N", top_n=4, window=63, slippage=0.0005, borrow_fee=0.02/252):
    universe = TNICUniverse(tnic_data_path="data/raw/tnic2_data.txt", target_year=2021)
    RIC_TO_GVKEY = {"XOM.N": "4503", "CVX.N": "5903", "COP.N": "61971", "OXY.N": "25964", "MPC.N": "294524"}
    GVKEY_TO_RIC = {v: k for k, v in RIC_TO_GVKEY.items()}
    
    raw_peer_rics, raw_weights = universe.get_top_n_peers(target_ric, RIC_TO_GVKEY, GVKEY_TO_RIC, n_peers=top_n)
    valid_indices = [i for i, ric in enumerate(raw_peer_rics) if ric != target_ric]
    peer_rics = [raw_peer_rics[i] for i in valid_indices]
    weights = np.array([raw_weights[i] for i in valid_indices])
    weights = weights / np.sum(weights) 
    
    rd.open_session()
    try:
        # We also pull the S&P 500 (.SPX) for Performance Attribution later
        df = rd.get_history(universe=[target_ric, "SPY"] + peer_rics, fields=["TRDPRC_1"], 
                            start="2021-01-01", end="2024-01-01", interval="1D")
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        df = df.ffill().bfill().dropna().astype(float)
        
        target_prices = df[target_ric].values
        peer_prices = df[peer_rics].values
        market_prices = df["SPY"].values
        dates = df.index
        T = len(target_prices)
        
        synthetic_index = universe.construct_peer_index(weights, peer_prices)
        governor = MidTermGovernor(dt=1/252)
        spread = governor.fit_kalman_cointegration(target_prices, synthetic_index)
        
        hjb_solver = HJBOptimalStopping(z_min=-10.0, z_max=10.0, grid_points=1000)
        pnl = np.zeros(T)
        signals = np.zeros(T)
        daily_returns = np.zeros(T)
        
        current_pos = 0 
        
        for t in range(window, T):
            local_spread = spread[t-window:t]
            
            # The Ergodicity Kill Switch (ADF Test)
            try:
                adf_stat = adfuller(local_spread)[1]
                is_ergodic = adf_stat < 0.05
            except:
                is_ergodic = False

            mu, sigma, half_life = governor.fit_rolling_ou_process(local_spread)
            
            if is_ergodic and (0 < half_life < 252):
                theta = np.log(2) / half_life
                total_friction_bps = (slippage * 2) + (borrow_fee * half_life)
                
                try:
                    bound_dist = abs(hjb_solver.solve_entry_boundary(mu=0.0, theta=theta, sigma=sigma, friction=total_friction_bps))
                except:
                    bound_dist = np.inf
                    
                hjb_upper = mu + bound_dist
                hjb_lower = mu - bound_dist
            else:
                # If regime breaks, mathematically forbid entry and force liquidation
                hjb_upper = np.inf
                hjb_lower = -np.inf

            # P&L Update
            if current_pos != 0:
                d_target = target_prices[t] - target_prices[t-1]
                d_peer = synthetic_index[t] - synthetic_index[t-1]
                profit = current_pos * (d_target - governor.beta_t[t-1] * d_peer)
                
                exposure = (governor.beta_t[t-1]*synthetic_index[t-1] if current_pos==1 else target_prices[t-1])
                profit -= exposure * borrow_fee
                pnl[t] = pnl[t-1] + profit
                denominator = target_prices[t-1] + abs(governor.beta_t[t-1]*synthetic_index[t-1])
                daily_returns[t] = profit / denominator if denominator > 0 else 0
            else:
                pnl[t] = pnl[t-1]

            # State Machine
            prev_pos = current_pos
            if not is_ergodic:
                current_pos = 0 # Force exit if random walk detected
            else:
                if current_pos == 0:
                    if spread[t] < hjb_lower: current_pos = 1
                    elif spread[t] > hjb_upper: current_pos = -1
                elif (current_pos == 1 and spread[t] >= mu) or (current_pos == -1 and spread[t] <= mu):
                    current_pos = 0
            
            # Slippage deduction
            if current_pos != prev_pos:
                cost = abs(current_pos - prev_pos) * (target_prices[t] + abs(governor.beta_t[t]*synthetic_index[t])) * slippage
                pnl[t] -= cost
                
            signals[t] = current_pos

        # Calculate Market Returns for Attribution
        market_returns = np.diff(market_prices) / market_prices[:-1]
        market_returns = np.insert(market_returns, 0, 0)
            
        return dates, pnl, signals, daily_returns, market_returns, window
    finally:
        rd.close_session()

# Execute Backtest
dates, pnl, signals, daily_returns, market_returns, window = run_master_backtest()

Streaming TNIC database for year 2021...
Loaded 1371773 peer connections for 2021.
Dynamically mapped 4 executable TNIC peers for XOM.N.
[MLE Calibration] Structural State Noise (Q): 2.25e-05 | Observation Noise (R): 8.63e-03


### 3. Performance Attribution (Jensen's Alpha)
To prove our dynamical system generates structural edge rather than leveraged beta, we regress our strategy's daily returns against the S&P 500 market proxy ($R_{strat} = \alpha + \beta R_{market}$). A statistically significant $\alpha$ with a near-zero $\beta$ confirms orthogonal isolation from standard risk premia.

In [8]:
# Extract active days (where we held a position)
active_idx = np.where(daily_returns != 0)[0]
strat_active = daily_returns[active_idx]
market_active = market_returns[active_idx]

# OLS Regression
X = sm.add_constant(market_active)
ols_model = sm.OLS(strat_active, X).fit()

print("=== Performance Attribution (CAPM Regression) ===")
print(ols_model.summary())

total_return = (pnl[-1] / 100000) * 100 # Assuming roughly 100k notional
sharpe = (np.mean(strat_active) / np.std(strat_active)) * np.sqrt(252) if len(strat_active)>0 else 0
print(f"\nFinal P&L: ${pnl[-1]:.2f}")
print(f"Annualized Active Sharpe Ratio: {sharpe:.2f}")

=== Performance Attribution (CAPM Regression) ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.006
Method:                 Least Squares   F-statistic:                   0.03980
Date:                Thu, 05 Mar 2026   Prob (F-statistic):              0.842
Time:                        18:09:18   Log-Likelihood:                 706.95
No. Observations:                 172   AIC:                            -1410.
Df Residuals:                     170   BIC:                            -1404.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
co

In [ ]:
# Extract active days and align dates
active_idx = np.where(daily_returns != 0)[0]
strat_active = daily_returns[active_idx]
active_dates = dates[active_idx].strftime('%Y-%m-%d')

# Fetch Fama-French 5-Factor Daily Data
ff5_dict = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2021-01-01', end='2024-01-01')
ff5 = ff5_dict[0] / 100.0 # Convert from percentage to decimals
ff5.index = ff5.index.strftime('%Y-%m-%d')

# Align the strategy returns with the FF5 factors using the exact active dates
aligned_data = pd.DataFrame({'Strategy': strat_active}, index=active_dates)
aligned_data = aligned_data.join(ff5, how='inner').dropna()

# OLS Multiple Regression
y = aligned_data['Strategy'] - aligned_data['RF']
X = sm.add_constant(aligned_data[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']])
ols_model = sm.OLS(y, X).fit()

print("=== Fama-French 5-Factor Performance Attribution ===")
print(ols_model.summary())

### 4. Market Microstructure & T-Cost Sensitivity
Because statistical arbitrage relies on crossing the bid-ask spread ($dP/dQ > 0$) to capture mean-reversion, strategy capacity is physically limited. Below, we empirically stress-test the algorithm against a vector of execution slippage (1 bps to 15 bps) to find the exact point of mathematical ruin.

In [ ]:
slippage_grid = np.linspace(0.0001, 0.0015, 10) # 1 bps to 15 bps
terminal_pnls = []

print("Running Friction Sensitivity Matrix...")
for slip in slippage_grid:
    _, temp_pnl, _, _, _, _ = run_master_backtest(slippage=slip)
    terminal_pnls.append(temp_pnl[-1])

fig = go.Figure(data=go.Scatter(x=slippage_grid*10000, y=terminal_pnls, mode='lines+markers'))
fig.update_layout(title="Execution Friction Threshold (T-Cost Sensitivity)",
                  xaxis_title="Slippage per Execution (Basis Points)",
                  yaxis_title="Terminal P&L ($)", template="plotly_white")
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

Running Friction Sensitivity Matrix...
Streaming TNIC database for year 2021...
Loaded 1371773 peer connections for 2021.
Dynamically mapped 4 executable TNIC peers for XOM.N.
[MLE Calibration] Structural State Noise (Q): 2.25e-05 | Observation Noise (R): 8.63e-03
